In [1]:
print('nurc-tts')

nurc-tts


In [ ]:
!pip install datasets

In [2]:
from datasets import load_dataset
import pandas as pd

In [7]:
splits = ['sao_paulo', 'recife']
all_metadata = []

for split_name in splits:
    ds = load_dataset('sidleal/nurc_tts_24khz', split=split_name)
    metadata = ds.remove_columns(['audio'])
    for entry in metadata:
        entry['split'] = split_name
        all_metadata.append(entry)

df = pd.DataFrame(all_metadata)

print(f"Total metadata rows loaded: {len(df)}")
display(df.head())

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/78 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/111 [00:00<?, ?it/s]

Total metadata rows loaded: 320916


,inquiry,segment,speaker,duration,text,inq_type,inq_recording_year,inq_gender,inq_quality,inq_themes,inq_age_group,split
0,SP_D2_015,SP_D2_015_SPEAKER_1_3_539.293_544.071.wav,SPEAKER 1,4.778,e cinco litros. o alqueire de são paulo? qual ...,D2,1971,M e M,Ótima,"Vegetais, agricultura Animais, rebanho",II e II,sao_paulo
1,SP_D2_015,SP_D2_015_SPEAKER_1_5_898.716_903.438.wav,SPEAKER 1,4.722,"muito bom. quais são as fases, todas as fases ...",D2,1971,M e M,Ótima,"Vegetais, agricultura Animais, rebanho",II e II,sao_paulo
2,SP_D2_015,SP_D2_015_SPEAKER_1_7_2346.528_2351.128.wav,SPEAKER 1,4.600,como é que ele cuida do umbigo do piseiro?,D2,1971,M e M,Ótima,"Vegetais, agricultura Animais, rebanho",II e II,sao_paulo
3,SP_D2_015,SP_D2_015_SPEAKER_1_11_3028.218_3031.298.wav,SPEAKER 1,3.080,e essa máquina costuma ser da própria fazenda ...,D2,1971,M e M,Ótima,"Vegetais, agricultura Animais, rebanho",II e II,sao_paulo
4,SP_D2_015,SP_D2_015_SPEAKER_1_15_3562.448_3566.068.wav,SPEAKER 1,3.620,"tipo moca, isso tem algum... não, moca não não...",D2,1971,M e M,Ótima,"Vegetais, agricultura Animais, rebanho",II e II,sao_paulo


In [8]:
unique_chars = set(''.join(df['text'].dropna().astype(str)))
sorted_chars = sorted(list(unique_chars))
print("Unique characters found:")
print(sorted_chars)

Unique characters found:
['\n', ' ', '%', ',', '-', '.', '/', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'á', 'â', 'ã', 'ç', 'é', 'ê', 'í', 'ó', 'ô', 'õ', 'ú']


#clean

In [ ]:
!pip install datasets huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
ds = load_dataset('sidleal/nurc_tts_24khz')
print(ds)

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/78 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/111 [00:00<?, ?it/s]

DatasetDict({
    sao_paulo: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 120621
    })
    recife: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 200295
    })
})


In [7]:
import re
def clean_text(batch):
    cleaned_texts = []
    
    for text in batch["text"]:
        if text is None:
            cleaned_texts.append("")
            continue
            
        cleaned = re.sub(r'[\n%/]', '', text) 
        cleaned_texts.append(cleaned)
        
    return {"text": cleaned_texts}


In [8]:

#Apply the transformation using .map()
# batched=True processes the data in chunks, making it much faster
cleaned_dataset = ds.map(clean_text, batched=True)


Map:   0%|          | 0/120621 [00:00<?, ? examples/s]

Map:   0%|          | 0/200295 [00:00<?, ? examples/s]

In [16]:
print(cleaned_dataset)

DatasetDict({
    sao_paulo: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 120621
    })
    recife: Dataset({
        features: ['inquiry', 'segment', 'speaker', 'duration', 'text', 'inq_type', 'inq_recording_year', 'inq_gender', 'inq_quality', 'inq_themes', 'inq_age_group', 'audio'],
        num_rows: 200295
    })
})


In [15]:
metadata = cleaned_dataset['sao_paulo'].remove_columns(['audio'])
dfx = pd.DataFrame(metadata)

unique_chars = set(''.join(dfx['text'].dropna().astype(str)))
sorted_chars = sorted(list(unique_chars))
print("Unique characters found:")
print(sorted_chars)

Unique characters found:
[' ', ',', '-', '.', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'á', 'â', 'ã', 'ç', 'é', 'ê', 'í', 'ó', 'ô', 'õ', 'ú']


In [17]:

# Upload it back to the Hugging Face Hub
cleaned_dataset.push_to_hub("sidleal/nurc_tts_24khz")

Uploading the dataset shards:   0%|          | 0/81 [00:00<?, ? shards/s]

Map:   0%|          | 0/1490 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/416M [00:00<?, ?B/s]

Map:   0%|          | 0/1490 [00:00<?, ? examples/s]

...

Uploading...:   0%|          | 0.00/380M [00:00<?, ?B/s]

Map:   0%|          | 0/1756 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/365M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/sidleal/nurc_tts_24khz/commit/16d12c6cf261ee4329e1d46f9d60b493ab39465d', commit_message='Upload dataset', commit_description='', oid='16d12c6cf261ee4329e1d46f9d60b493ab39465d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sidleal/nurc_tts_24khz', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sidleal/nurc_tts_24khz'), pr_revision=None, pr_num=None)